#Quantum Pipelines for Wound Healing Classification

Two quantum approaches to the same donor-grouped wound-healing classification task used in the classical notebook:

- **Variational Quantum Feature Selection (VQFS):** a shallow parameterized quantum circuit learns per-gene importance weights on top of four different classical prefilters (Mutual Information, mRMR, Variance, Random), compared against a classical MI-only baseline.
- **Quantum Kernel SVM:** genes are selected with CPSS (Complementary Pairs Stability Selection) + mRMR, then an amplitude-embedding fidelity quantum kernel feeds an SVM, benchmarked against a classical RBF-kernel SVM on the same genes.

Both paths share data loading, preprocessing, scaling, and the classifier registry, defined once below.

## Load Libraries

In [ ]:
!pip install -q scanpy anndata pandas==2.2.2 numpy scipy scikit-learn scikit-misc mlflow mlxtend dagshub joblib torch pennylane
!pip install -q pennylane-lightning
!pip install -q gseapy

## Import Libraries

In [ ]:
import copy
import gc
import json
import os
import time
import warnings
from functools import partial

import anndata
import dagshub
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pennylane as qml
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim

from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    StratifiedKFold,
    StratifiedShuffleSplit,
    cross_val_score,
    train_test_split,
)
from sklearn.svm import SVC

## Load Filtered Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

adata = sc.read_h5ad('/content/drive/.../final_dataset/bio_dataset.h5ad')
adata

### Remove `healing_stage` and `batch` from the dataset

Both columns could leak the label to the model (see Classical notebook for the full rationale), so they are removed and stored separately for later analysis.

In [ ]:
healing_stage_categories = adata.obs['healing_stage'].copy()
adata.obs.drop(columns=['healing_stage'], inplace=True)

batchs = adata.obs['batch'].copy()
adata.obs.drop(columns=['batch'], inplace=True)

adata

## Experiment Tracking with MLflow using DagsHub



In [ ]:
USER_NAME = ""
REPO_NAME = ""
TOKEN = os.environ.get("DAGSHUB_TOKEN", "")

os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = TOKEN
os.environ['MLFLOW_TRACKING_URI'] = f"https://dagshub.com/{USER_NAME}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

## Group K-Fold Implementation


In [ ]:
X_full = adata.X
y_full = adata.obs['healing_state'].values
donor_groups = adata.obs['donor_id'].values

outer_cv = GroupKFold(n_splits=4)

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_full, y_full, donor_groups)):
    test_donor = np.unique(donor_groups[test_idx])[0]
    train_donors = np.unique(donor_groups[train_idx])
    print(f"\n{'='*60}")
    print(f"OUTER FOLD {fold_idx + 1}/4: Testing on Donor {test_donor}")
    print(f"Training on Donors: {train_donors}")
    print(f"{'='*60}")

## Scaling Methodology


In [ ]:
def scale_train_test(adata_train, adata_test, max_value=10):
    sc.pp.scale(adata_train, max_value=max_value)

    train_mean = adata_train.var['mean'].values
    train_std = adata_train.var['std'].values
    train_std_safe = np.where(train_std == 0, 1.0, train_std)

    if sparse.issparse(adata_test.X):
        X_test = adata_test.X.toarray()
    else:
        X_test = np.array(adata_test.X)

    X_test_scaled = (X_test - train_mean) / train_std_safe
    X_test_scaled = np.clip(X_test_scaled, -max_value, max_value)
    adata_test.X = X_test_scaled

    adata_test.uns['scaling_info'] = {
        'mean': train_mean,
        'std': train_std_safe,
        'max_value': max_value,
        'n_zero_var_genes': int((train_std == 0).sum())
    }
    if (train_std == 0).sum() > 0:
        print(f"  \u26a0 Warning: {(train_std == 0).sum()} genes had zero variance in training set")

    return adata_train, adata_test

## Model Registry & Selection

In [ ]:
MODEL_REGISTRY = {
    'SVM-RBF': {
        'estimator': SVC(kernel='rbf', probability=True, random_state=42),
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'gamma': [0.001, 0.01, 0.1, 1],
            'class_weight': ['balanced']
        },
        'n_jobs': 1
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 300],
            'max_depth': [20, None],
            'class_weight': ['balanced'],
            'min_samples_split': [2, 10]
        },
        'n_jobs': 1
    },
    'LogisticRegression': {
        'estimator': LogisticRegression(random_state=42),
        'param_grid': {
            'C': np.logspace(-2, 2, 10),
            'penalty': ['l1', 'l2'],
            'solver': ['saga'],
            'class_weight': ['balanced'],
            'max_iter': [10000],
            'tol': [1e-3]
        },
        'n_jobs': 1
    }
}

# CHANGE THIS VARIABLE to switch models
SELECTED_MODEL = 'LogisticRegression'

if SELECTED_MODEL not in MODEL_REGISTRY:
    raise ValueError(f"Model '{SELECTED_MODEL}' not found.")

model_config = MODEL_REGISTRY[SELECTED_MODEL]
print(f"\u2713 Selected Model: {SELECTED_MODEL}")
print(f"\u2713 Configuration loaded.")

---
# Variational Quantum Feature Selection (VQFS)

A shallow parameterized quantum circuit (angle embedding + a few entangling layers) is trained to weight genes, using Complementary-Pairs Stability Selection (CPSS) over many random splits to find genes that are stably important, rather than important on one lucky split. Four classical prefilters narrow the gene set to `n_qubits` genes before the quantum stage runs (Mutual Information, mRMR, Variance, Random), plus a classical MI-only baseline for comparison.

### Classical prefilter methods

Each prefilter cuts the ~2,000 highly-variable genes down to `n_qubits` candidates before the quantum circuit is trained on them.

In [ ]:
def mrmr_prefilter(X, y, k, mi_relevance=None):
    n_genes = X.shape[1]
    if mi_relevance is None:
        mi_relevance = mutual_info_classif(X, y)

    corr_matrix = np.abs(np.corrcoef(X, rowvar=False))
    np.nan_to_num(corr_matrix, copy=False, nan=0.0)

    selected = [int(np.argmax(mi_relevance))]
    remaining = list(range(n_genes))
    remaining.remove(selected[0])

    while len(selected) < k:
        redundancy = corr_matrix[np.ix_(remaining, selected)].mean(axis=1)
        mrmr_score = mi_relevance[remaining] - redundancy
        best_pos = int(np.argmax(mrmr_score))
        best = remaining[best_pos]
        selected.append(best)
        remaining.pop(best_pos)

    return np.array(selected)


def variance_prefilter(X, y, k, precomputed_variance=None):
    """
    If precomputed_variance is provided, rank by that instead of X.var(axis=0).
    This matters because X here is typically the z-scored/clipped matrix
    (post sc.pp.scale), where every gene has variance ~= 1 by construction --
    ranking on that destroys the biological variance signal entirely.
    Pass in variance computed on the log-normalized, PRE-scaling matrix instead.
    """
    if precomputed_variance is not None:
        gene_variances = precomputed_variance
    else:
        gene_variances = X.var(axis=0)
    return np.argsort(gene_variances)[::-1][:k]


def random_prefilter(X, y, k, random_state=42):
    rng = np.random.RandomState(random_state)
    return rng.choice(X.shape[1], size=k, replace=False)

### Quantum VQFS architecture

> **Barren-plateau guard:** `n_entangler_layers` is capped at 2 in `VQFS_PQK_Network`. At 15 qubits with local PauliZ observables, 1–2 shallow entangling layers stays well clear of the barren-plateau regime; do not raise this past 2 without re-validating gradient magnitudes.

> **`l1_penalty=1e-4`** across all four VQFS arms is intentional: the prior value (0.05) was crushing `feature_weights` to zero early in training (a \"dead-qubit\" trap). Inspect `selection_probability_` after a few runs, if it shows almost no sparsity at all, nudge `l1_penalty` up toward `1e-3` gradually rather than jumping back to `0.05`.

In [ ]:
class VQFS_PQK_Network(nn.Module):
    _device_warning_shown = False  # class-level flag so the warning prints once, not 24x/fold

    def __init__(self, n_qubits: int, n_entangler_layers: int = 1):
        super().__init__()
        if n_entangler_layers > 2:
            print(f"\u26a0 WARNING: n_entangler_layers={n_entangler_layers} exceeds the "
                  f"recommended barren-plateau-safe ceiling of 2 for {n_qubits} qubits. "
                  f"Gradients may vanish. Proceeding anyway, but consider reducing depth.")

        self.n_qubits = n_qubits
        self.feature_weights = nn.Parameter(torch.ones(n_qubits))
        self.entangler_weights = nn.Parameter(0.01 * torch.randn(n_entangler_layers, n_qubits))
        self.probe = nn.Linear(n_qubits, 1)

        # Device must be set up BEFORE the qnode decorator references self.dev.
        try:
            self.dev = qml.device("lightning.qubit", wires=n_qubits)
            if not VQFS_PQK_Network._device_warning_shown:
                print(f"[VQFS_PQK_Network] Using lightning.qubit ({n_qubits} qubits)")
                VQFS_PQK_Network._device_warning_shown = True
        except Exception as e:
            self.dev = qml.device("default.qubit", wires=n_qubits)
            print(f"\u26a0\u26a0\u26a0 lightning.qubit unavailable ({e}) -- FALLING BACK to default.qubit. "
                  f"This will be dramatically slower. Fix the install before running the full grid.")

        @qml.qnode(self.dev, interface="torch", diff_method="adjoint")
        def pqk_circuit(inputs, f_weights, e_weights):
            scaled_inputs = inputs * f_weights
            qml.AngleEmbedding(scaled_inputs, wires=range(n_qubits))
            qml.BasicEntanglerLayers(weights=e_weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qnode = pqk_circuit

    def forward(self, x):
        q_out = self.qnode(x, self.feature_weights, self.entangler_weights)
        q_out_tensor = torch.stack(q_out, dim=1).to(dtype=torch.float32)
        return self.probe(q_out_tensor)


class SklearnQuantumVQFSWrapper(BaseEstimator, TransformerMixin):
    """
    Variational Quantum Feature Selection via CPSS (Complementary Pairs Stability Selection).

    prefilter_method: 'mi' | 'mrmr' | 'variance' | 'random'

    Pipeline: prefilter to n_qubits genes -> train many small VQC sub-models on
    complementary random splits -> keep genes whose |feature_weight| survives above
    the fold median in a stable fraction (>= stability_threshold) of sub-models.
    """
    def __init__(self, n_qubits=15, target_features=10, cpss_splits=8, epochs=20,
                 lr=0.05, l1_penalty=1e-4, random_state=42, batch_size=64,
                 stability_threshold=0.6, n_entangler_layers=1, prefilter_method='mi',
                 max_vqc_samples=300):
        self.n_qubits = n_qubits
        self.target_features = target_features
        self.cpss_splits = cpss_splits
        self.epochs = epochs
        self.lr = lr
        self.l1_penalty = l1_penalty
        self.random_state = random_state
        self.batch_size = batch_size
        self.stability_threshold = stability_threshold
        self.n_entangler_layers = n_entangler_layers
        self.prefilter_method = prefilter_method
        self.max_vqc_samples = max_vqc_samples

        self.selected_indices_ = None
        self.top_genes_ = None
        self.selection_probability_ = None
        self.stage_timings_ = None
        self.precomputed_gene_variance = None  # set externally before fit() for the 'variance' arm
        self.device = torch.device("cpu")

    def _run_prefilter(self, X, y):
        if self.prefilter_method == 'mi':
            mi_scores = mutual_info_classif(X, y, random_state=self.random_state)
            pre_selected_indices = np.argsort(mi_scores)[::-1][:self.n_qubits]
        elif self.prefilter_method == 'mrmr':
            pre_selected_indices = mrmr_prefilter(X, y, self.n_qubits)
        elif self.prefilter_method == 'variance':
            pre_selected_indices = variance_prefilter(
                X, y, self.n_qubits, precomputed_variance=self.precomputed_gene_variance
            )
        elif self.prefilter_method == 'random':
            pre_selected_indices = random_prefilter(X, y, self.n_qubits, self.random_state)
        else:
            raise ValueError(f"Unknown prefilter_method: {self.prefilter_method}")

        return X[:, pre_selected_indices], pre_selected_indices

    def fit(self, X, y=None, feature_names=None):
        if y is None:
            raise ValueError(
                "VQFS is a supervised quantum feature selection method and requires target labels 'y'. "
                "Please extract y_train earlier in your loop and pass it as: fit_transform(X, y=y_train)"
            )

        fit_start = time.time()
        print(f"\n[VQFS:{self.prefilter_method}] Stage 1: Prefiltering "
              f"({X.shape[1]} -> {self.n_qubits} genes)...")
        stage1_start = time.time()
        X_pre, pre_selected_indices = self._run_prefilter(X, y)
        pre_selected_genes = (np.array(feature_names)[pre_selected_indices]
                               if feature_names is not None else None)
        stage1_time = time.time() - stage1_start

        n_sub_models_expected = self.cpss_splits * 2

        # Subsampling: without it, Stage 2 trains on the full fold (thousands of cells)
        # which, at ~0.17s/circuit-eval, makes each sub-model take hours, not minutes.
        n_available = X_pre.shape[0]
        if n_available > self.max_vqc_samples:
            print(f"[VQFS:{self.prefilter_method}] Subsampling {n_available} -> "
                  f"{self.max_vqc_samples} cells for quantum training "
                  f"(stratified, random_state={self.random_state})")
            subsample_splitter = StratifiedShuffleSplit(
                n_splits=1, train_size=self.max_vqc_samples, random_state=self.random_state
            )
            sub_idx, _ = next(subsample_splitter.split(X_pre, y))
            X_pre = X_pre[sub_idx]
            y = y[sub_idx]
        else:
            print(f"[VQFS:{self.prefilter_method}] {n_available} cells available "
                  f"(<= max_vqc_samples={self.max_vqc_samples}), no subsampling needed")

        # Upfront time estimate: benchmark one sub-model before committing to the full loop.
        print(f"[VQFS:{self.prefilter_method}] Benchmarking one VQC training run to estimate total time...")
        bench_start = time.time()
        sss_bench = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=self.random_state)
        bench_idx_A, bench_idx_B = next(sss_bench.split(X_pre, y))

        X_bench = torch.FloatTensor(X_pre[bench_idx_A]).to(self.device)
        y_bench = torch.FloatTensor(y[bench_idx_A]).unsqueeze(1).to(self.device)

        torch.manual_seed(self.random_state)
        bench_model = VQFS_PQK_Network(self.n_qubits, self.n_entangler_layers).to(self.device)
        bench_optimizer = optim.Adam(bench_model.parameters(), lr=self.lr)
        bench_criterion = nn.BCEWithLogitsLoss()

        bench_model.train()
        n_bench_samples = X_bench.shape[0]
        for epoch in range(self.epochs):
            permutation = torch.randperm(n_bench_samples)
            for i in range(0, n_bench_samples, self.batch_size):
                b_idx = permutation[i:i + self.batch_size]
                xb, yb = X_bench[b_idx], y_bench[b_idx]
                bench_optimizer.zero_grad()
                logits = bench_model(xb)
                loss = bench_criterion(logits, yb) + self.l1_penalty * torch.norm(bench_model.feature_weights, 1)
                loss.backward()
                bench_optimizer.step()

        bench_elapsed = time.time() - bench_start
        estimated_total_stage2 = bench_elapsed * n_sub_models_expected
        print(f"[VQFS:{self.prefilter_method}] Benchmark: 1 sub-model took {bench_elapsed:.1f}s "
              f"({n_bench_samples} samples, {self.epochs} epochs)")
        print(f"[VQFS:{self.prefilter_method}] ESTIMATED Stage 2 total: "
              f"~{estimated_total_stage2/60:.1f} min for {n_sub_models_expected} sub-models "
              f"(~{estimated_total_stage2/self.cpss_splits:.1f}s/split)")

        del bench_model, bench_optimizer, X_bench, y_bench, logits, loss
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Actual CPSS loop.
        stage2_start = time.time()
        selection_frequencies = np.zeros(self.n_qubits)
        n_sub_models = 0
        sub_model_times = []

        sss = StratifiedShuffleSplit(n_splits=self.cpss_splits, test_size=0.5,
                                      random_state=self.random_state)
        for split_idx, (idx_A, idx_B) in enumerate(sss.split(X_pre, y)):
            split_start = time.time()
            for sub_idx, idx in enumerate([idx_A, idx_B]):
                X_sub = torch.FloatTensor(X_pre[idx]).to(self.device)
                y_sub = torch.FloatTensor(y[idx]).unsqueeze(1).to(self.device)

                torch.manual_seed(self.random_state + split_idx * 2 + sub_idx)
                model = VQFS_PQK_Network(self.n_qubits, self.n_entangler_layers).to(self.device)
                optimizer = optim.Adam(model.parameters(), lr=self.lr)
                criterion = nn.BCEWithLogitsLoss()

                model.train()
                n_samples = X_sub.shape[0]
                for epoch in range(self.epochs):
                    permutation = torch.randperm(n_samples)
                    for i in range(0, n_samples, self.batch_size):
                        b_idx = permutation[i:i + self.batch_size]
                        xb, yb = X_sub[b_idx], y_sub[b_idx]
                        optimizer.zero_grad()
                        logits = model(xb)
                        bce_loss = criterion(logits, yb)
                        l1_loss = self.l1_penalty * torch.norm(model.feature_weights, 1)
                        total_loss = bce_loss + l1_loss
                        total_loss.backward()
                        optimizer.step()

                # All post-training inspection wrapped in no_grad -- prevents the computational
                # graph from this sub-model's forward passes from being retained past this point.
                with torch.no_grad():
                    abs_weights = torch.abs(model.feature_weights).cpu().numpy()
                    threshold = np.median(abs_weights)
                    survivors = abs_weights >= threshold
                    selection_frequencies += survivors.astype(int)
                    n_sub_models += 1

                # Explicit teardown + gc.collect() every sub-model (not just every split) --
                # with 24 sub-models per fold this is the difference between a clean run and
                # an OOM crash around split 10-12.
                del model, optimizer, X_sub, y_sub, logits, total_loss, bce_loss, l1_loss
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            split_elapsed = time.time() - split_start
            sub_model_times.append(split_elapsed)
            if (split_idx + 1) % 4 == 0 or split_idx == self.cpss_splits - 1:
                avg_split_time = np.mean(sub_model_times)
                remaining_splits = self.cpss_splits - (split_idx + 1)
                eta_min = (remaining_splits * avg_split_time) / 60
                print(f"  [VQFS:{self.prefilter_method}] split {split_idx+1}/{self.cpss_splits} "
                      f"done in {split_elapsed:.1f}s (avg {avg_split_time:.1f}s/split) -- "
                      f"ETA {eta_min:.1f} min remaining")

        stage2_time = time.time() - stage2_start

        print(f"[VQFS:{self.prefilter_method}] Stage 3: Consolidating Final Quantum Biomarkers...")
        stage3_start = time.time()
        selection_probability = selection_frequencies / n_sub_models
        self.selection_probability_ = selection_probability

        stable_mask = selection_probability >= self.stability_threshold
        stable_relative_idx = np.where(stable_mask)[0]

        if len(stable_relative_idx) >= self.target_features:
            ranked = stable_relative_idx[np.argsort(selection_probability[stable_relative_idx])[::-1]]
            final_relative_indices = ranked[:self.target_features]
        else:
            print(f"[VQFS:{self.prefilter_method}] Warning: only {len(stable_relative_idx)} genes "
                  f"cleared stability_threshold={self.stability_threshold}. Falling back to top "
                  f"{self.target_features} by raw selection frequency.")
            final_relative_indices = np.argsort(selection_frequencies)[::-1][:self.target_features]

        self.selected_indices_ = pre_selected_indices[final_relative_indices]

        if feature_names is not None:
            self.top_genes_ = {"Quantum_Selected_Genes": list(pre_selected_genes[final_relative_indices]),
                                "prefilter_method": self.prefilter_method}
            print(f"  -> Selected Genes ({self.prefilter_method}): "
                  f"{self.top_genes_['Quantum_Selected_Genes']}")
            print(f"  -> Selection probabilities: "
                  f"{np.round(selection_probability[final_relative_indices], 2)}")

        stage3_time = time.time() - stage3_start
        total_time = time.time() - fit_start

        self.stage_timings_ = {
            "prefilter_seconds": stage1_time,
            "cpss_stability_selection_seconds": stage2_time,
            "consolidation_seconds": stage3_time,
            "total_fit_seconds": total_time,
            "avg_seconds_per_vqc_split": float(np.mean(sub_model_times)) if sub_model_times else None,
        }
        print(f"[VQFS:{self.prefilter_method}] Fit complete in {total_time/60:.1f} min "
              f"(prefilter={stage1_time:.1f}s, CPSS={stage2_time/60:.1f}min, "
              f"consolidation={stage3_time:.1f}s)")
        return self

    def transform(self, X, y=None):
        return X[:, self.selected_indices_]

### Classical MI-only baseline

In [ ]:
class ClassicalMIBaselineWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, target_features=10, random_state=42):
        self.target_features = target_features
        self.random_state = random_state
        self.selector = None
        self.selected_indices_ = None
        self.top_genes_ = None

    def fit(self, X, y=None, feature_names=None):
        if y is None:
            raise ValueError("ClassicalMIBaselineWrapper requires y for supervised selection.")

        # Pin mutual_info_classif's internal randomness so the baseline gene set is
        # reproducible run-to-run, matching the VQFS 'mi' arm.
        mi_scorer = partial(mutual_info_classif, random_state=self.random_state)
        self.selector = SelectKBest(score_func=mi_scorer, k=self.target_features)
        self.selector.fit(X, y)
        self.selected_indices_ = self.selector.get_support(indices=True)

        if feature_names is not None:
            self.top_genes_ = {"Classical_MI_Genes": list(np.array(feature_names)[self.selected_indices_])}
            print(f"  -> Classical baseline genes: {self.top_genes_['Classical_MI_Genes']}")
        return self

    def transform(self, X, y=None):
        return X[:, self.selected_indices_]

### Feature Extractor Registry (VQFS)

In [ ]:
FEATURE_EXTRACTOR_REGISTRY = {
    'VQFS_PQK_MI': {
        'extractor': SklearnQuantumVQFSWrapper(
            n_qubits=15, target_features=10, cpss_splits=8, epochs=20,
            l1_penalty=1e-4, batch_size=64, stability_threshold=0.6,
            n_entangler_layers=1, prefilter_method='mi', max_vqc_samples=300
        )
    },
    'VQFS_PQK_mRMR': {
        'extractor': SklearnQuantumVQFSWrapper(
            n_qubits=15, target_features=10, cpss_splits=8, epochs=20,
            l1_penalty=1e-4, batch_size=64, stability_threshold=0.6,
            n_entangler_layers=1, prefilter_method='mrmr', max_vqc_samples=300
        )
    },
    'VQFS_PQK_Variance': {
        'extractor': SklearnQuantumVQFSWrapper(
            n_qubits=15, target_features=10, cpss_splits=8, epochs=20,
            l1_penalty=1e-4, batch_size=64, stability_threshold=0.6,
            n_entangler_layers=1, prefilter_method='variance', max_vqc_samples=300
        )
    },
    'VQFS_PQK_Random': {
        'extractor': SklearnQuantumVQFSWrapper(
            n_qubits=15, target_features=10, cpss_splits=8, epochs=20,
            l1_penalty=1e-4, batch_size=64, stability_threshold=0.6,
            n_entangler_layers=1, prefilter_method='random', max_vqc_samples=300
        )
    },
    'MI_Classical_Baseline': {
        'extractor': ClassicalMIBaselineWrapper(target_features=10)
    }
}

### Run VQFS Pipeline


In [ ]:
def run_already_logged(experiment_name, extractor_name, model_name, donor):
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        return False
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{extractor_name}_{model_name}_Donor_{donor}'"
    )
    return len(runs) > 0


os.environ["SCIPY_ARRAY_API"] = "1"

EXTRACTORS_TO_RUN = ['VQFS_PQK_MI', 'VQFS_PQK_mRMR', 'VQFS_PQK_Variance',
                      'VQFS_PQK_Random', 'MI_Classical_Baseline']

mlflow.set_experiment("")

for SELECTED_EXTRACTOR in EXTRACTORS_TO_RUN:
    if SELECTED_EXTRACTOR not in FEATURE_EXTRACTOR_REGISTRY:
        raise ValueError(f"Extractor '{SELECTED_EXTRACTOR}' not found.")

    print(f"\n{'#'*70}\n# RUNNING EXTRACTOR: {SELECTED_EXTRACTOR}\n{'#'*70}")

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_full, y_full, groups=donor_groups)):
        test_donor = np.unique(donor_groups[test_idx])[0]

        if run_already_logged("", SELECTED_EXTRACTOR, SELECTED_MODEL, test_donor):
            print(f"Skipping {SELECTED_EXTRACTOR} / Donor {test_donor} (already logged in MLflow)")
            continue

        fold_start = time.time()
        feature_extractor = copy.deepcopy(FEATURE_EXTRACTOR_REGISTRY[SELECTED_EXTRACTOR]['extractor'])
        print(f"\nFold {fold_idx+1}: Processing Test Donor {test_donor}...")

        # --- PREPROCESSING ---
        adata_train = adata[train_idx].copy()
        adata_test = adata[test_idx].copy()
        adata_train.X = adata_train.X.astype('float32')
        adata_test.X = adata_test.X.astype('float32')

        sc.pp.normalize_total(adata_train, target_sum=1e4)
        sc.pp.log1p(adata_train)
        sc.pp.normalize_total(adata_test, target_sum=1e4)
        sc.pp.log1p(adata_test)

        # 'seurat' (not 'seurat_v3'): seurat_v3 expects raw counts, but adata_train is
        # already log-normalized at this point.
        sc.pp.highly_variable_genes(adata_train, flavor='seurat', n_top_genes=2000)
        hvg_genes = adata_train.var[adata_train.var['highly_variable']].index
        adata_train = adata_train[:, hvg_genes].copy()
        adata_test = adata_test[:, hvg_genes].copy()

        # Snapshot gene variance on the log-normalized, UNSCALED matrix, before
        # scale_train_test z-scores everything to variance ~= 1. This is what the
        # 'variance' prefilter arm should actually rank on.
        X_train_prescale = (adata_train.X.toarray()
                             if sparse.issparse(adata_train.X) else np.array(adata_train.X))
        gene_variance_unscaled = X_train_prescale.var(axis=0)
        del X_train_prescale

        adata_train, adata_test = scale_train_test(adata_train, adata_test, max_value=10)

        y_train = adata_train.obs['healing_state'].values
        y_test = adata_test.obs['healing_state'].values

        print(f"Extracting features using: {SELECTED_EXTRACTOR}...")
        X_train_input = adata_train.X.toarray() if sparse.issparse(adata_train.X) else adata_train.X
        X_test_input = adata_test.X.toarray() if sparse.issparse(adata_test.X) else adata_test.X

        # Hand the pre-scale variance to the variance-prefilter arm only.
        if SELECTED_EXTRACTOR == 'VQFS_PQK_Variance':
            feature_extractor.precomputed_gene_variance = gene_variance_unscaled

        x_train = feature_extractor.fit_transform(X_train_input, y=y_train, feature_names=hvg_genes)
        x_test = feature_extractor.transform(X_test_input)

        # --- MODEL TRAINING ---
        inner_cv = GroupKFold(n_splits=3)
        inner_groups = adata_train.obs['donor_id'].values

        grid = GridSearchCV(
            estimator=model_config['estimator'],
            param_grid=model_config['param_grid'],
            cv=inner_cv.split(x_train, y_train, groups=inner_groups),
            scoring='f1',
            n_jobs=1,
            verbose=0
        )
        grid.fit(x_train, y_train)
        best_model = grid.best_estimator_

        train_pred = best_model.predict(x_train)
        y_pred = best_model.predict(x_test)
        y_proba = best_model.predict_proba(x_test)[:, 1]

        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()
        train_f1 = float(f1_score(y_train, train_pred))
        test_f1 = float(f1_score(y_test, y_pred))

        metrics = {
            'train_f1': train_f1,
            'test_f1': test_f1,
            'balanced_acc': float(balanced_accuracy_score(y_test, y_pred)),
            'roc_auc': float(roc_auc_score(y_test, y_proba)),
            'overfit_gap': train_f1 - test_f1,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        }

        elapsed_min = (time.time() - fold_start) / 60
        print(f"  cv_f1={grid.best_score_:.3f} | test_f1={metrics['test_f1']:.3f} | "
              f"roc_auc={metrics['roc_auc']:.3f} | bal_acc={metrics['balanced_acc']:.3f} | "
              f"fold_runtime={elapsed_min:.1f} min")

        # --- MLFLOW LOGGING ---
        with mlflow.start_run(run_name=f"{SELECTED_EXTRACTOR}_{SELECTED_MODEL}_Donor_{test_donor}"):
            mlflow.log_param("feature_extractor", SELECTED_EXTRACTOR)
            mlflow.log_param("classifier", SELECTED_MODEL)
            mlflow.log_param("test_donor", str(test_donor))
            mlflow.log_param("n_selected_genes", x_train.shape[1])

            mlflow.log_metric("cv_f1_score", grid.best_score_)
            mlflow.log_metrics({
                "test_f1_score": metrics['test_f1'],
                "train_f1_score": metrics['train_f1'],
                "balanced_accuracy": metrics['balanced_acc'],
                "roc_auc": metrics['roc_auc'],
                "overfit_gap": metrics['overfit_gap'],
                "tn": metrics['tn'], "fp": metrics['fp'],
                "fn": metrics['fn'], "tp": metrics['tp'],
            })
            mlflow.log_metric("fold_runtime_minutes", elapsed_min)

            if hasattr(feature_extractor, 'stage_timings_') and feature_extractor.stage_timings_ is not None:
                mlflow.log_metrics({
                    "prefilter_seconds": feature_extractor.stage_timings_["prefilter_seconds"],
                    "cpss_stability_selection_seconds": feature_extractor.stage_timings_["cpss_stability_selection_seconds"],
                    "consolidation_seconds": feature_extractor.stage_timings_["consolidation_seconds"],
                    "total_extractor_fit_seconds": feature_extractor.stage_timings_["total_fit_seconds"],
                })
                if feature_extractor.stage_timings_["avg_seconds_per_vqc_split"] is not None:
                    mlflow.log_metric("avg_seconds_per_vqc_split",
                                       feature_extractor.stage_timings_["avg_seconds_per_vqc_split"])

            if hasattr(feature_extractor, 'top_genes_') and feature_extractor.top_genes_ is not None:
                mlflow.log_dict(feature_extractor.top_genes_, "selected_genes.json")

            if hasattr(feature_extractor, 'selection_probability_') and feature_extractor.selection_probability_ is not None:
                mlflow.log_dict(
                    {"selection_probability": feature_extractor.selection_probability_.tolist()},
                    "vqfs_selection_probability.json"
                )

            results = {
                'test_donor': str(test_donor),
                'feature_extractor': SELECTED_EXTRACTOR,
                'classifier': SELECTED_MODEL,
                'cv_f1': float(grid.best_score_),
                **metrics,
                'confusion_matrix': cm.tolist(),
                'fold_runtime_minutes': elapsed_min,
                'stage_timings': (feature_extractor.stage_timings_
                                  if hasattr(feature_extractor, 'stage_timings_') else None),
                'selected_genes': (feature_extractor.top_genes_ if hasattr(feature_extractor, 'top_genes_') else None),
            }
            filename = f'pathB_{SELECTED_EXTRACTOR}_{test_donor}.json'
            with open(filename, 'w') as f_out:
                json.dump(results, f_out, indent=2, default=str)
            mlflow.log_artifact(filename)

        # --- MEMORY CLEANUP ---
        del adata_train, adata_test, X_train_input, X_test_input, x_train, x_test
        del grid, best_model, feature_extractor
        gc.collect()
        print(f"\u2713 Fold {test_donor} complete ({SELECTED_EXTRACTOR}) in {elapsed_min:.1f} min. Memory cleared.")

### Summary Report

Pulls every run back from MLflow and produces a five-arm comparison report (extractor-level summary, win counts, gene-set Jaccard overlap, aggregated gene selection, per-donor breakdown).

In [ ]:
from itertools import combinations
from collections import Counter

experiment_name = ""
classifier_name = "LogisticRegression"
baseline_extractor = "MI_Classical_Baseline"
quantum_extractors = ["VQFS_PQK_MI", "VQFS_PQK_mRMR", "VQFS_PQK_Variance", "VQFS_PQK_Random"]
all_extractors = quantum_extractors + [baseline_extractor]
excluded_donors = []
output_file = f"/content/drive/.../PathB_Quantum_vs_Classical_Report_{classifier_name}.txt"

client = mlflow.tracking.MlflowClient()
exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    raise SystemExit(f"Experiment '{experiment_name}' not found!")

print(f"Fetching runs for {experiment_name}...")
df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
df = df[df['params.classifier'] == classifier_name]
if excluded_donors:
    df = df[~df['params.test_donor'].isin(excluded_donors)]
df = df.dropna(subset=['metrics.test_f1_score'])

if df.empty:
    raise SystemExit("No valid runs found matching the criteria.")
print(f"Found {len(df)} valid runs. Generating report...")


def get_top_genes(run_id):
    try:
        path = client.download_artifacts(run_id, "selected_genes.json")
        with open(path) as f:
            data = json.load(f)
        for key in ("Quantum_Selected_Genes", "Classical_MI_Genes"):
            if key in data:
                return data[key]
        return list(data.values())[0] if data else []
    except Exception:
        return []


df['top_genes'] = df['run_id'].apply(get_top_genes)

metric_cols = {
    'test_f1_score': 'Test F1',
    'balanced_accuracy': 'Balanced Acc',
    'roc_auc': 'ROC-AUC',
    'overfit_gap': 'Overfit Gap',
    'fold_runtime_minutes': 'Runtime (min)',
}
for col in metric_cols:
    if f'metrics.{col}' not in df.columns:
        df[f'metrics.{col}'] = np.nan

pivot_f1 = df.pivot_table(index='params.test_donor', columns='params.feature_extractor',
                           values='metrics.test_f1_score')

# Win-count ranking: best test_f1 per donor.
win_counts = Counter()
for donor in pivot_f1.index:
    row = pivot_f1.loc[donor].dropna()
    if not row.empty:
        win_counts[row.idxmax()] += 1

# Extractor-level summary across donors.
summary_rows = []
for extractor in all_extractors:
    sub = df[df['params.feature_extractor'] == extractor]
    if sub.empty:
        continue
    row = {'extractor': extractor, 'n_donors': len(sub)}
    for col, label in metric_cols.items():
        vals = sub[f'metrics.{col}'].dropna()
        row[label + ' mean'] = vals.mean() if len(vals) else np.nan
        row[label + ' std'] = vals.std() if len(vals) > 1 else 0.0
        row[label + ' range'] = (vals.max() - vals.min()) if len(vals) > 1 else 0.0
    row['wins'] = win_counts.get(extractor, 0)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('extractor')


def jaccard(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return np.nan
    return len(a & b) / len(a | b) if (a | b) else np.nan


jaccard_records = []
for e1, e2 in combinations(all_extractors, 2):
    scores = []
    for donor in df['params.test_donor'].unique():
        g1 = df[(df['params.feature_extractor'] == e1) & (df['params.test_donor'] == donor)]['top_genes']
        g2 = df[(df['params.feature_extractor'] == e2) & (df['params.test_donor'] == donor)]['top_genes']
        if not g1.empty and not g2.empty:
            j = jaccard(g1.iloc[0], g2.iloc[0])
            if not np.isnan(j):
                scores.append(j)
    if scores:
        jaccard_records.append({'pair': f"{e1} vs {e2}", 'mean_jaccard': np.mean(scores), 'n_donors': len(scores)})

jaccard_df = pd.DataFrame(jaccard_records).sort_values('mean_jaccard', ascending=False)

all_genes = []
for _, row in df[df['params.feature_extractor'].isin(quantum_extractors)].iterrows():
    all_genes.extend(row['top_genes'])
most_common_genes = Counter(all_genes).most_common()

with open(output_file, 'w') as f:
    f.write("="*70 + "\nPATH B: FULL FIVE-ARM COMPARISON REPORT (Quantum x4 + Classical)\n")
    f.write(f"Classifier: {classifier_name}\n" + "="*70 + "\n\n")

    f.write("--- EXTRACTOR-LEVEL SUMMARY (mean +/- range across donors; n too small for significance testing) ---\n\n")
    f.write(summary_df.round(4).to_string())
    f.write("\n\n")

    f.write("--- WIN COUNT (best test F1 per donor) ---\n")
    for extractor in all_extractors:
        f.write(f"  {extractor}: {win_counts.get(extractor, 0)} / {len(pivot_f1.index)} donors\n")
    f.write("\n")

    f.write("--- GENE SET OVERLAP (Jaccard similarity, averaged across shared donors) ---\n")
    f.write(jaccard_df.round(3).to_string(index=False) if not jaccard_df.empty else " (insufficient data)\n")
    f.write("\n\n")

    f.write("--- AGGREGATED GENE SELECTION (Quantum arms only) ---\n")
    for gene, count in most_common_genes:
        f.write(f"  * {gene}: selected in {count} (extractor, donor) run(s)\n")

    f.write("\n--- PER-DONOR BREAKDOWN ---\n\n")
    for donor in sorted(pivot_f1.index):
        f.write(f"DONOR {donor}\n" + "-"*20 + "\n")
        baseline_row = df[(df['params.test_donor'] == donor) &
                           (df['params.feature_extractor'] == baseline_extractor)]
        baseline_f1 = baseline_row['metrics.test_f1_score'].iloc[0] if not baseline_row.empty else np.nan

        for extractor in all_extractors:
            match = df[(df['params.test_donor'] == donor) & (df['params.feature_extractor'] == extractor)]
            if match.empty:
                f.write(f"  [{extractor}] -- no run found for this donor --\n\n")
                continue
            row = match.iloc[0]
            f.write(f"  [{extractor}]\n")
            f.write(f"    Top Genes: {', '.join(row['top_genes'])}\n")
            f.write(f"    Test F1 Score: {row['metrics.test_f1_score']:.4f}\n")
            f.write(f"    Balanced Accuracy: {row['metrics.balanced_accuracy']:.4f}\n")
            f.write(f"    ROC-AUC: {row['metrics.roc_auc']:.4f}\n")
            f.write(f"    Train F1 Score: {row.get('metrics.train_f1_score', np.nan):.4f}\n")
            f.write(f"    Overfit Gap: {row['metrics.overfit_gap']:.4f}\n")
            f.write(f"    Fold Runtime (min): {row['metrics.fold_runtime_minutes']:.2f}\n")
            if extractor != baseline_extractor and not pd.isna(baseline_f1):
                adv = row['metrics.test_f1_score'] - baseline_f1
                f.write(f"    Advantage vs Classical Baseline: {adv:+.4f}\n")
            f.write("\n")

    f.write("="*70 + "\n\n")
    f.write("NOTE: With n=4 donors and no classical-CPSS control arm yet implemented,\n")
    f.write("these results are descriptive only and should not be over-interpreted as\n")
    f.write("statistically significant quantum advantage.\n")

print(f"Report successfully generated: {output_file}")
print("\n=== Extractor Summary Preview ===")
print(summary_df[[c for c in summary_df.columns if 'mean' in c or c == 'wins']].round(4))

---
# Quantum Kernel SVM (CPSS + mRMR)

Genes are pre-screened by global Mutual Information, then narrowed further with Complementary-Pairs Stability Selection (CPSS) using a fast mRMR criterion inside each pair. The resulting genes feed both an amplitude-embedding fidelity quantum kernel and a classical RBF kernel, each paired with an SVM, so the two can be benchmarked head-to-head on identical features.

### Minimum Redundancy Maximum Relevance (fast mRMR)

In [ ]:
def fast_mrmr_selection(X_subset, subset_mi_scores, n_candidates=64):
    n_prescreened = X_subset.shape[1]

    # Calculate full correlation matrix once (fast).
    corr_matrix = np.corrcoef(X_subset, rowvar=False)
    # Square correlation to approximate Mutual Information bounds.
    corr_matrix = np.nan_to_num(corr_matrix ** 2, nan=0.0)

    selected_local_idx = []
    remaining = set(range(n_prescreened))

    # 1. Start with the gene with the highest MI in this subset.
    best_start = np.argmax(subset_mi_scores)
    selected_local_idx.append(best_start)
    remaining.remove(best_start)

    # 2. Greedy selection loop.
    while len(selected_local_idx) < n_candidates:
        best_score = -np.inf
        best_gene = None
        for gene_idx in remaining:
            relevance = subset_mi_scores[gene_idx]
            # Fast redundancy check: max correlation with any already-selected gene.
            redundancy = corr_matrix[gene_idx, selected_local_idx].max()
            mrmr_score = relevance - redundancy
            if mrmr_score > best_score:
                best_score = mrmr_score
                best_gene = gene_idx
        selected_local_idx.append(best_gene)
        remaining.remove(best_gene)

    return selected_local_idx


print("\u2713 Fast MRMR function defined.")

### Checkpoint Manager

Saves kernel matrices and intermediate results to Drive automatically, so a Colab disconnect doesn't force recomputing an expensive `NxN` quantum kernel from scratch.

In [ ]:
import pickle
from datetime import datetime

PROJECT_DIR = '/content/drive/MyDrive/QML_WoundHealing'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR = f'{PROJECT_DIR}/results'

for d in [PROJECT_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)


class CheckpointManager:
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = checkpoint_dir
        self.log_file = f'{checkpoint_dir}/execution_log.json'
        self.log = self._load_log()

    def _load_log(self):
        if os.path.exists(self.log_file):
            with open(self.log_file, 'r') as f:
                return json.load(f)
        return {}

    def _save_log(self):
        with open(self.log_file, 'w') as f:
            json.dump(self.log, f, indent=2)

    def exists(self, key):
        return key in self.log and os.path.exists(self.log[key]['path'])

    def save(self, key, data, description=""):
        path = f'{self.checkpoint_dir}/{key}.pkl'
        with open(path, 'wb') as f:
            pickle.dump(data, f)
        self.log[key] = {
            'path': path,
            'description': description,
            'timestamp': datetime.now().isoformat(),
            'size_mb': os.path.getsize(path) / 1e6
        }
        self._save_log()
        print(f"  \u2713 Saved: {key} ({self.log[key]['size_mb']:.1f} MB)")

    def load(self, key):
        with open(self.log[key]['path'], 'rb') as f:
            data = pickle.load(f)
        print(f"  \u2192 Loaded: {key}")
        return data

    def status(self):
        print("\n=== CHECKPOINT STATUS ===")
        if not self.log:
            print("No checkpoints found")
            return
        for key, info in self.log.items():
            exists = "\u2713" if os.path.exists(info['path']) else "\u2717 MISSING"
            print(f"  {exists} {key} ({info['timestamp'][:19]})")


ckpt = CheckpointManager(CHECKPOINT_DIR)
ckpt.status()

### PennyLane Quantum Kernel Setup


In [ ]:
N_QUBITS = 6
N_TRAIN_SUBSAMPLE = 300
N_TEST_SUBSAMPLE = 200

dev = qml.device("lightning.qubit", wires=N_QUBITS)


@qml.qnode(dev, interface="numpy")
def kernel_circuit(x1, x2):
    qml.AmplitudeEmbedding(x1, wires=range(N_QUBITS), normalize=True)
    qml.adjoint(qml.AmplitudeEmbedding)(x2, wires=range(N_QUBITS), normalize=True)
    return qml.probs(wires=range(N_QUBITS))


def quantum_kernel(x1, x2):
    return float(kernel_circuit(x1, x2)[0])


def pad_and_normalize(X, n_qubits):
    required = 2 ** n_qubits
    if X.shape[1] < required:
        X = np.pad(X, ((0, 0), (0, required - X.shape[1])), mode='constant')
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return X / norms


def build_kernel_matrix(X1, X2, kernel_fn, label=""):
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    is_sym = X1 is X2
    t_start = time.time()

    for i in range(n1):
        if is_sym:
            K[i, i] = 1.0
        start_j = i + 1 if is_sym else 0
        for j in range(start_j, n2):
            val = kernel_fn(X1[i], X2[j])
            K[i, j] = val
            if is_sym:
                K[j, i] = val
        if (i + 1) % 25 == 0:
            elapsed_min = (time.time() - t_start) / 60
            remaining = (n1 - i - 1) * (elapsed_min / (i + 1))
            print(f"  {label} row {i+1}/{n1} -- "
                  f"{elapsed_min:.1f}min elapsed, ~{remaining:.1f}min remaining")
    return K


def tune_C(estimator_fn, X_or_K, y, groups, cv, C_grid, label):
    best_C, best_f1 = C_grid[0], -1
    print(f"\nTuning C -- {label}")
    for C_val in C_grid:
        model = estimator_fn(C_val)
        scores = cross_val_score(
            model, X_or_K, y,
            cv=cv.split(X_or_K, y, groups=groups),
            scoring='f1', n_jobs=-1
        )
        print(f"  C={C_val:<8}: F1={scores.mean():.3f} \u00b1 {scores.std():.3f}")
        if scores.mean() > best_f1:
            best_f1 = scores.mean()
            best_C = C_val
    return best_C, best_f1


def fit_and_evaluate(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    train_pred = model.predict(X_train)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'train_f1': float(f1_score(y_train, train_pred)),
        'test_f1': float(f1_score(y_test, y_pred)),
        'balanced_acc': float(balanced_accuracy_score(y_test, y_pred)),
        'roc_auc': float(roc_auc_score(y_test, y_proba)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'confusion_matrix': cm.tolist()
    }

### Run Quantum Kernel Pipeline



In [ ]:
mlflow.set_experiment("")

folds = list(outer_cv.split(X_full, y_full, groups=donor_groups))

for fold_idx, (train_idx, test_idx) in enumerate(folds):
    test_donor = np.unique(donor_groups[test_idx])[0]
    train_donors = np.unique(donor_groups[train_idx])

    if run_already_logged("",
                           "QuantumKernel_CPSS_mRMR", "SVM", test_donor):
        print(f"\nSkipping Donor {test_donor} (already logged in MLflow)")
        continue

    fold_start = time.time()
    print(f"\n{'='*60}")
    print(f"OUTER FOLD {fold_idx + 1}/4: Testing on Donor {test_donor}")
    print(f"Training on Donors: {train_donors}")
    print(f"{'='*60}")

    # ---- PREPROCESSING ----
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()

    sc.pp.normalize_total(adata_train, target_sum=1e4)
    sc.pp.log1p(adata_train)
    sc.pp.normalize_total(adata_test, target_sum=1e4)
    sc.pp.log1p(adata_test)

    adata_train, adata_test = scale_train_test(adata_train, adata_test, max_value=10)

    X_train_scaled = adata_train.X if not sparse.issparse(adata_train.X) else adata_train.X.toarray()
    X_test_scaled = adata_test.X if not sparse.issparse(adata_test.X) else adata_test.X.toarray()
    y_train = adata_train.obs['healing_state'].values
    y_test = adata_test.obs['healing_state'].values

    # ---- CPSS + mRMR FEATURE SELECTION ----
    n_prescreen = 500
    print(f"Calculating Global MI for {X_train_scaled.shape[1]} genes...")
    global_mi = mutual_info_classif(X_train_scaled, y_train, random_state=42, n_neighbors=5)
    top_indices = np.argsort(global_mi)[::-1][:n_prescreen]
    X_prescreened = X_train_scaled[:, top_indices]
    local_mi_scores = global_mi[top_indices]
    print(f"\u2713 Pre-screening complete. Shape: {X_prescreened.shape}")

    B_pairs = 30
    n_candidates = 64
    stability_threshold = 0.6
    rng = np.random.RandomState(42)
    selection_counts = np.zeros(X_train_scaled.shape[1])

    class0_idx = np.where(y_train == 0)[0]
    class1_idx = np.where(y_train == 1)[0]
    half_class0 = len(class0_idx) // 2
    half_class1 = len(class1_idx) // 2

    print(f"Starting CPSS loop ({B_pairs} pairs)...")
    for b in range(B_pairs):
        c0_perm = rng.permutation(class0_idx)
        c1_perm = rng.permutation(class1_idx)

        A_idx = np.concatenate([c0_perm[:half_class0], c1_perm[:half_class1]])
        Ac_idx = np.concatenate([c0_perm[half_class0:2*half_class0], c1_perm[half_class1:2*half_class1]])

        selected_A_local = fast_mrmr_selection(X_prescreened[A_idx], local_mi_scores, n_candidates)
        selected_Ac_local = fast_mrmr_selection(X_prescreened[Ac_idx], local_mi_scores, n_candidates)

        for local_idx in set(selected_A_local) & set(selected_Ac_local):
            selection_counts[top_indices[local_idx]] += 1

        if (b + 1) % 10 == 0:
            print(f"  -> Completed pair {b+1}/{B_pairs}")

    selection_frequency = selection_counts / B_pairs
    stable_gene_idx = np.where(selection_frequency >= stability_threshold)[0]
    stable_gene_idx = stable_gene_idx[np.argsort(selection_frequency[stable_gene_idx])[::-1]]
    if len(stable_gene_idx) > n_candidates:
        stable_gene_idx = stable_gene_idx[:n_candidates]
    final_quantum_genes = stable_gene_idx
    print(f"\u2713 CPSS complete. Selected {len(final_quantum_genes)} genes.")

    # ---- DATA PREPARATION + BENCHMARK ----
    x_train_64 = X_train_scaled[:, final_quantum_genes]
    x_test_64 = X_test_scaled[:, final_quantum_genes]
    gene_names = list(adata.var_names[final_quantum_genes])
    print(f"Quantum gene matrix -- train: {x_train_64.shape}, test: {x_test_64.shape}")

    x_bench = pad_and_normalize(x_train_64[:2], N_QUBITS)
    t = time.time()
    for _ in range(20):
        quantum_kernel(x_bench[0], x_bench[1])
    ms_per_eval = (time.time() - t) / 20 * 1000

    n_train_pairs = N_TRAIN_SUBSAMPLE * (N_TRAIN_SUBSAMPLE - 1) // 2
    n_test_pairs = N_TEST_SUBSAMPLE * N_TRAIN_SUBSAMPLE
    print(f"Per kernel eval: {ms_per_eval:.2f}ms")
    print(f"K_train estimate: {n_train_pairs * ms_per_eval / 1000 / 60:.1f} min")
    print(f"K_test estimate: {n_test_pairs * ms_per_eval / 1000 / 60:.1f} min")
    print(f"Total estimate: {(n_train_pairs + n_test_pairs) * ms_per_eval / 1000 / 60:.1f} min")

    # ---- QUANTUM KERNEL COMPUTATION ----
    train_sub_idx, _ = train_test_split(
        np.arange(len(x_train_64)), train_size=N_TRAIN_SUBSAMPLE,
        stratify=y_train, random_state=42
    )
    x_train_sub = pad_and_normalize(x_train_64[train_sub_idx], N_QUBITS)
    y_train_sub = y_train[train_sub_idx]
    sub_donor_groups = adata_train.obs['donor_id'].values[train_sub_idx]

    test_sub_idx, _ = train_test_split(
        np.arange(len(x_test_64)), train_size=N_TEST_SUBSAMPLE,
        stratify=y_test, random_state=42
    )
    x_test_sub = pad_and_normalize(x_test_64[test_sub_idx], N_QUBITS)
    y_test_eval = y_test[test_sub_idx]

    # Include sample sizes in the checkpoint key so old sizes don't overwrite new runs.
    k_train_key = f'fold_{test_donor}_K_train_{N_TRAIN_SUBSAMPLE}'
    k_test_key = f'fold_{test_donor}_K_test_{N_TEST_SUBSAMPLE}x{N_TRAIN_SUBSAMPLE}'

    if ckpt.exists(k_train_key):
        K_train_q = ckpt.load(k_train_key)
        print("K_train loaded from checkpoint")
    else:
        print(f"Computing K_train ({N_TRAIN_SUBSAMPLE}\u00d7{N_TRAIN_SUBSAMPLE})...")
        K_train_q = build_kernel_matrix(x_train_sub, x_train_sub, quantum_kernel, "K_train")
        ckpt.save(k_train_key, K_train_q, f"K_train donor {test_donor} (N={N_TRAIN_SUBSAMPLE})")

    if ckpt.exists(k_test_key):
        K_test_q = ckpt.load(k_test_key)
        print("K_test loaded from checkpoint")
    else:
        print(f"Computing K_test ({N_TEST_SUBSAMPLE}\u00d7{N_TRAIN_SUBSAMPLE})...")
        K_test_q = build_kernel_matrix(x_test_sub, x_train_sub, quantum_kernel, "K_test")
        ckpt.save(k_test_key, K_test_q, f"K_test donor {test_donor} (N={N_TEST_SUBSAMPLE})")

    min_eig = np.linalg.eigvalsh(K_train_q).min()
    if min_eig < -1e-6:
        print(f"\u26a0 Applying PSD correction (min eigenvalue: {min_eig:.4f})")
        K_train_q += (-min_eig + 1e-6) * np.eye(len(K_train_q))
    print(f"K_train: {K_train_q.shape} | K_test: {K_test_q.shape}")

    # ---- C TUNING ON PRECOMPUTED KERNEL ----
    C_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
    inner_cv = GroupKFold(n_splits=3)

    best_C_q, best_cv_f1_q = tune_C(
        estimator_fn=lambda C: SVC(kernel='precomputed', C=C,
                                    class_weight='balanced', probability=True, random_state=42),
        X_or_K=K_train_q, y=y_train_sub, groups=sub_donor_groups,
        cv=inner_cv, C_grid=C_grid, label="Quantum Kernel SVM"
    )
    best_C_c, best_cv_f1_c = tune_C(
        estimator_fn=lambda C: SVC(kernel='rbf', C=C, gamma='scale',
                                    class_weight='balanced', probability=True, random_state=42),
        X_or_K=x_train_64[train_sub_idx], y=y_train_sub, groups=sub_donor_groups,
        cv=inner_cv, C_grid=C_grid, label="Classical RBF SVM (same genes)"
    )
    print(f"\nBest C -- Quantum: {best_C_q} (F1={best_cv_f1_q:.3f}) | "
          f"Classical: {best_C_c} (F1={best_cv_f1_c:.3f})")

    # ---- FINAL MODELS AND METRICS ----
    qsvm = SVC(kernel='precomputed', C=best_C_q, class_weight='balanced',
               probability=True, random_state=42)
    q_metrics = fit_and_evaluate(qsvm, K_train_q, y_train_sub, K_test_q, y_test_eval)

    csvm = SVC(kernel='rbf', C=best_C_c, gamma='scale', class_weight='balanced',
               probability=True, random_state=42)
    c_metrics = fit_and_evaluate(
        csvm, x_train_64[train_sub_idx], y_train_sub, x_test_64[test_sub_idx], y_test_eval
    )

    quantum_advantage = q_metrics['test_f1'] - c_metrics['test_f1']
    print(f"\n{'='*50}")
    print(f"Quantum SVM   -- Test F1: {q_metrics['test_f1']:.3f} | ROC-AUC: {q_metrics['roc_auc']:.3f}")
    print(f"Classical RBF -- Test F1: {c_metrics['test_f1']:.3f} | ROC-AUC: {c_metrics['roc_auc']:.3f}")
    print(f"Quantum Advantage: {quantum_advantage:+.3f}")

    if quantum_advantage > 0.05:
        hypothesis = 'SUPPORTED'
    elif quantum_advantage < -0.05:
        hypothesis = 'REFUTED'
    else:
        hypothesis = 'INCONCLUSIVE'
    print(f"Hypothesis: {hypothesis}")

    # ---- MLFLOW LOGGING ----
    with mlflow.start_run(run_name=f"QuantumKernel_CPSS_mRMR_SVM_Donor_{test_donor}"):
        mlflow.log_params({
            "path": "",
            "feature_selector": "CPSS_mRMR",
            "test_donor": str(test_donor),
            "encoding": "amplitude",
            "n_qubits": N_QUBITS,
            "n_quantum_genes": len(final_quantum_genes),
            "n_train_subsample": N_TRAIN_SUBSAMPLE,
            "n_test_subsample": N_TEST_SUBSAMPLE,
            "n_train_full": len(x_train_64),
            "n_test_full": len(x_test_64),
            "cpss_b_pairs": B_pairs,
            "cpss_threshold": stability_threshold,
            "quantum_best_C": best_C_q,
            "classical_best_C": best_C_c,
            "imbalance_ratio": round(float(y_train.sum() / len(y_train)), 3),
            "gene_names_top10": str(gene_names[:10])
        })

        mlflow.log_metrics({
            "quantum_cv_f1": best_cv_f1_q,
            "quantum_train_f1": q_metrics['train_f1'],
            "quantum_test_f1": q_metrics['test_f1'],
            "quantum_bal_acc": q_metrics['balanced_acc'],
            "quantum_roc_auc": q_metrics['roc_auc'],
            "quantum_overfit": q_metrics['train_f1'] - q_metrics['test_f1'],
            "quantum_tn": q_metrics['tn'], "quantum_fp": q_metrics['fp'],
            "quantum_fn": q_metrics['fn'], "quantum_tp": q_metrics['tp']
        })
        mlflow.log_metrics({
            "classical_cv_f1": best_cv_f1_c,
            "classical_train_f1": c_metrics['train_f1'],
            "classical_test_f1": c_metrics['test_f1'],
            "classical_bal_acc": c_metrics['balanced_acc'],
            "classical_roc_auc": c_metrics['roc_auc'],
            "classical_overfit": c_metrics['train_f1'] - c_metrics['test_f1'],
            "classical_tn": c_metrics['tn'], "classical_fp": c_metrics['fp'],
            "classical_fn": c_metrics['fn'], "classical_tp": c_metrics['tp']
        })
        mlflow.log_metric("quantum_advantage_f1", quantum_advantage)

        for rank, (gene, freq) in enumerate(zip(gene_names, selection_frequency[final_quantum_genes])):
            mlflow.log_metric(f"gene_{rank+1:02d}_stability", float(freq))

        results = {
            'test_donor': test_donor,
            'quantum': {**q_metrics, 'best_C': best_C_q, 'cv_f1': best_cv_f1_q},
            'classical_baseline': {**c_metrics, 'best_C': best_C_c, 'cv_f1': best_cv_f1_c},
            'quantum_advantage': float(quantum_advantage),
            'hypothesis': hypothesis,
            'selected_genes': gene_names,
            'gene_stability': [float(f) for f in selection_frequency[final_quantum_genes]]
        }
        filename = f'pathB_results_{test_donor}.json'
        with open(filename, 'w') as f_out:
            json.dump(results, f_out, indent=2)
        mlflow.log_artifact(filename)

        elapsed_total = (time.time() - fold_start) / 60
        mlflow.log_metric("fold_runtime_minutes", elapsed_total)
        print(f"\u2713 Donor {test_donor} logged to MLflow ({elapsed_total:.1f} min total)")

---
## Supplementary Visualizations: Quantum vs. Classical Benchmark



In [ ]:
!pip install -q pandas seaborn matplotlib

### Aggregated consensus gene selection across donors

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

gene_counts = {
    'LGALS3': 3, 'CFD': 2, 'AHNAK': 2, 'CYB5R3': 2, 'LAMP1': 2, 'GSN': 2,
    'NOP53': 1, 'PDIA3': 1, 'SELENOW': 1, 'PCBP2': 1, 'SURF4': 1,
    'STAT2': 1, 'SDF4': 1, 'TMED7': 1, 'QSOX1': 1
}

df_genes = pd.DataFrame(list(gene_counts.items()), columns=['Gene', 'Donor_Count'])

plt.figure(figsize=(10, 6), dpi=300)
sns.set_theme(style="whitegrid")

colors = ['#1f77b4' if c == 3 else '#aec7e8' if c == 2 else '#d3d3d3' for c in df_genes['Donor_Count']]

ax = sns.barplot(data=df_genes, y='Gene', x='Donor_Count', hue='Gene', palette=colors,
                  legend=False, edgecolor="black", linewidth=0.8)

plt.title("Aggregated Consensus Gene Selection Across Donors", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Donors Selected In", fontsize=12, labelpad=10)
plt.ylabel("Gene Marker", fontsize=12)
plt.xticks([0, 1, 2, 3])
plt.xlim(0, 3.5)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f"{int(width)} donor(s)", (width + 0.08, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=10,
                    fontweight='bold' if width == 3 else 'normal',
                    color='#1f77b4' if width == 3 else 'black')

sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig("gene_selection_frequency.png", dpi=300, bbox_inches='tight')
plt.show()

### Test F1 and ROC-AUC across donors: Quantum vs. Classical

In [ ]:
data = [
    {'Donor': 'D1', 'Model': 'Quantum', 'Test F1': 0.8750, 'ROC-AUC': 0.7307},
    {'Donor': 'D1', 'Model': 'Classical', 'Test F1': 0.8750, 'ROC-AUC': 0.8512},
    {'Donor': 'D2', 'Model': 'Quantum', 'Test F1': 0.9437, 'ROC-AUC': 0.2449},
    {'Donor': 'D2', 'Model': 'Classical', 'Test F1': 0.9541, 'ROC-AUC': 0.8955},
    {'Donor': 'D3', 'Model': 'Quantum', 'Test F1': 0.0000, 'ROC-AUC': 0.3049},
    {'Donor': 'D3', 'Model': 'Classical', 'Test F1': 0.9265, 'ROC-AUC': 0.8626},
    {'Donor': 'D4', 'Model': 'Quantum', 'Test F1': 0.9275, 'ROC-AUC': 0.6084},
    {'Donor': 'D4', 'Model': 'Classical', 'Test F1': 0.9576, 'ROC-AUC': 0.8833},
]
df_bench = pd.DataFrame(data)
colors = {'Quantum': '#6A1B9A', 'Classical': '#1565C0'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300, sharey=True)
sns.set_theme(style="whitegrid")

sns.barplot(data=df_bench, x='Donor', y='Test F1', hue='Model', palette=colors, ax=axes[0], edgecolor='black')
axes[0].set_title("Test F1 Score Across Donors", fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Metric Score", fontsize=11)
for p in axes[0].patches:
    height = p.get_height()
    if height >= 0:
        axes[0].annotate(f"{height:.2f}",
                          (p.get_x() + p.get_width() / 2., height / 2 if height > 0.15 else height + 0.02),
                          ha='center', va='center', fontsize=9,
                          color='white' if height > 0.15 else 'black', fontweight='bold')

sns.barplot(data=df_bench, x='Donor', y='ROC-AUC', hue='Model', palette=colors, ax=axes[1], edgecolor='black')
axes[1].set_title("ROC-AUC Score Across Donors", fontsize=13, fontweight='bold')
for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f"{height:.2f}",
                          (p.get_x() + p.get_width() / 2., height / 2 if height > 0.15 else height + 0.02),
                          ha='center', va='center', fontsize=9,
                          color='white' if height > 0.15 else 'black', fontweight='bold')

for ax in axes:
    ax.set_xlabel("Donor ID", fontsize=11)
    sns.despine(ax=ax, top=True, right=True)

plt.suptitle("Quantum vs. Classical Benchmark across Donors", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("donor_performance_gap.png", dpi=300, bbox_inches='tight')
plt.show()

### Generalization dynamics: overfit gap vs. test performance

In [ ]:
scatter_data = [
    {'Donor': 'D1', 'Model': 'Quantum', 'Test_F1': 0.8750, 'Overfit_Gap': 0.0989},
    {'Donor': 'D1', 'Model': 'Classical', 'Test_F1': 0.8750, 'Overfit_Gap': 0.1239},
    {'Donor': 'D2', 'Model': 'Quantum', 'Test_F1': 0.9437, 'Overfit_Gap': -0.0236},
    {'Donor': 'D2', 'Model': 'Classical', 'Test_F1': 0.9541, 'Overfit_Gap': 0.0459},
    {'Donor': 'D3', 'Model': 'Quantum', 'Test_F1': 0.0000, 'Overfit_Gap': 0.0000},
    {'Donor': 'D3', 'Model': 'Classical', 'Test_F1': 0.9265, 'Overfit_Gap': 0.0664},
    {'Donor': 'D4', 'Model': 'Quantum', 'Test_F1': 0.9275, 'Overfit_Gap': 0.0725},
    {'Donor': 'D4', 'Model': 'Classical', 'Test_F1': 0.9576, 'Overfit_Gap': 0.0424},
]
df_scatter = pd.DataFrame(scatter_data)

plt.figure(figsize=(9, 6), dpi=300)
sns.set_theme(style="whitegrid")

colors = {'Quantum': '#6A1B9A', 'Classical': '#1565C0'}
markers = {'Quantum': 'o', 'Classical': 's'}

for model_type, group in df_scatter.groupby('Model'):
    plt.scatter(group['Overfit_Gap'], group['Test_F1'], label=model_type,
                color=colors[model_type], marker=markers[model_type],
                s=140, edgecolor='black', linewidth=1.2, zorder=3)
    for _, row in group.iterrows():
        plt.annotate(f" {row['Donor']}", (row['Overfit_Gap'], row['Test_F1']),
                      xytext=(5, -2), textcoords='offset points', fontsize=10,
                      fontweight='bold', color=colors[model_type])

plt.axvline(0, color='gray', linestyle='--', linewidth=1, label='Zero Overfit Line')

plt.title("Generalization Dynamics: Overfit Gap vs. Test Performance", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Overfit Gap (Train F1 - Test F1)", fontsize=11)
plt.ylabel("Test F1 Score", fontsize=11)
plt.xlim(-0.05, 0.15)
plt.ylim(-0.05, 1.05)
plt.legend(title="Model Type", loc='center left', frameon=True)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig("overfit_gap_vs_performance.png", dpi=300, bbox_inches='tight')
plt.show()

### Results table

In [ ]:
data_table = [
    ["Donor D1", "Quantum Model", "0.8750", "0.6239", "0.7307", "0.0989"],
    ["", "Classical Baseline", "0.8750", "0.6239", "0.8512", "0.1239"],
    ["Donor D2", "Quantum Model", "0.9437", "0.5000", "0.2449", "-0.0236"],
    ["", "Classical Baseline", "0.9541", "0.7194", "0.8955", "0.0459"],
    ["Donor D3", "Quantum Model", "0.0000", "0.5000", "0.3049", "0.0000"],
    ["", "Classical Baseline", "0.9265", "0.6805", "0.8626", "0.0664"],
    ["Donor D4", "Quantum Model", "0.9275", "0.5455", "0.6084", "0.0725"],
    ["", "Classical Baseline", "0.9576", "0.5927", "0.8833", "0.0424"],
]
columns = ["Donor ID", "Classifier / Model", "Test F1", "Bal. Acc.", "ROC-AUC", "Overfit Gap"]

fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
fig.patch.set_facecolor('#020B14')
ax.set_facecolor('#020B14')
ax.axis('off')

BG_DARK = '#020B14'
BORDER_CYAN = '#005F73'
HEADER_CYAN = '#00E5FF'
TEXT_CYAN = '#33E0FF'
TEXT_WHITE = '#FFFFFF'

table = ax.table(cellText=data_table, colLabels=columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor(BORDER_CYAN)
    cell.set_linewidth(1.2)
    if row == 0:
        cell.set_facecolor('#051A2E')
        cell.get_text().set_color(HEADER_CYAN)
        cell.get_text().set_weight('bold')
        cell.get_text().set_fontsize(12)
    else:
        cell.set_facecolor(BG_DARK)
        if col == 0:
            cell.get_text().set_color(HEADER_CYAN)
            cell.get_text().set_weight('bold')
            cell.set_text_props(ha='left')
        elif col == 1:
            cell.get_text().set_color(TEXT_WHITE)
            cell.set_text_props(ha='left')
        else:
            cell.get_text().set_color(TEXT_CYAN)
            cell.get_text().set_weight('bold')

plt.tight_layout()
plt.savefig("quantum_classical_dark_table.png", dpi=300, facecolor=fig.get_facecolor(), bbox_inches='tight')
plt.show()